# Tâche 8: Export final du corpus

**Projet 1 · African Language Data & LLM Readiness · AfrikLang**

Combine le corpus normalisé (tâche 6) et la provenance finale avec quality_flags (tâche 7) en :
- `data/processed/corpus/<langue>.jsonl` texte seul (`{"text": ...}`), schéma validé à la tâche 2
- `data/processed/provenance/<langue>.jsonl` un fichier par langue, relié au corpus par empreinte SHA-256
- `data/processed/provenance.parquet` même contenu, en Parquet
- `data/processed/exclusions.jsonl` textes écartés à l'export (erreurs de collecte), avec leur raison
- `data/processed/export_exemples.jsonl` quelques exemples par langue pour relecture

Validation automatique : schéma JSON (bibliothèque jsonschema) sur chaque ligne écrite, correspondance exacte corpus <-> provenance par empreinte, quality_flags restreint aux 4 valeurs valides.

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\akota\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSES = ROOT / "data" / "processed" / "qualite" / "analyses"


def lancer(*args):
    # Lance src/exporter.py et affiche son journal au fur et à mesure.
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.Popen([sys.executable, "-u", str(ROOT / "src" / "exporter.py"), *args], cwd=ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        p.terminate()
        print("\nArrêté.")
    return p.wait()


def derniere(nom):
    # Dernière analyse d'un type donné (ici : export)
    return sorted(ANALYSES.glob(f"*_{nom}"))[-1]

## Lancer l'export

In [3]:
lancer()
d = derniere("export")
eb = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{eb['textes']} textes ; {eb['exportes']} exportés, {eb['exclus']} écarté(s) ; "
      f"drapeaux {eb['drapeaux']} ; {eb['duree_s']} s")
print("anomalies :", eb["anomalies"] or "aucune")
tab = pd.DataFrame([{"langue": r["langue"], "textes": r["textes"], "exportés": r["exportes"],
                     "écartés": r["exclus"], **r["drapeaux"]} for r in eb["langues"]]).set_index("langue").fillna(0)
tab

2026-11-22T08:13:23 INFO 8 Export final : 69 langue(s), 15 processus ; source=data\processed\normalise\20261113T141439Z, provenance=20261122T070453Z_ecriture_drapeaux
2026-11-22T08:13:30 INFO tn       7309 textes :     7309 exportés,  0 écarté(s), drapeaux {'near_duplicate': 17, 'language_mismatch': 163}  (3 s)
2026-11-22T08:13:32 INFO zgh     12192 textes :    12192 exportés,  0 écarté(s), drapeaux {'boilerplate': 55, 'near_duplicate': 31, 'language_mismatch': 308, 'template_generated': 5996}  (4 s)
2026-11-22T08:13:32 INFO so      15572 textes :    15572 exportés,  0 écarté(s), drapeaux {'language_mismatch': 399, 'near_duplicate': 19, 'template_generated': 82, 'boilerplate': 4}  (5 s)
2026-11-22T08:13:33 INFO dag     15711 textes :    15711 exportés,  0 écarté(s), drapeaux {'language_mismatch': 2987, 'near_duplicate': 17, 'template_generated': 882}  (6 s)
2026-11-22T08:13:34 INFO gpe      5873 textes :     5873 exportés,  0 écarté(s), drapeaux {'language_mismatch': 54}  (2 s)
2026-11

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\akota\\desktop\\projet_stage_nlp\\data\\processed\\qualite\\analyses\\20261122T081323Z_export\\bilan.json'

## Reprise après plantage (disque plein pendant le seul regroupement final)

Si `lancer()` plante avec `OSError: No space left on device` **après** que toutes les langues ont déjà été traitées (visible dans le journal : une ligne par langue déjà affichée), tout le travail par langue est déjà fait et sur disque (`corpus/` et les fragments de provenance). Pas besoin de relancer tout l'export : on reprend juste la finalisation, avec le nom du dossier d'export planté (visible dans le journal ou sous `data/processed/qualite/analyses/`).


In [3]:
# Remplacer par le nom du dossier d'export planté, ex. 20261122T081323Z_export
lancer("--terminer", "20261122T081323Z_export")
d = derniere("export")
eb = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{eb['textes']} textes ; {eb['exportes']} exportés, {eb['exclus']} écarté(s) ; "
      f"drapeaux {eb['drapeaux']} ; {eb['duree_s']} s")
print("anomalies :", eb["anomalies"] or "aucune")

2026-11-22T09:08:19 INFO 8 (reprise) Finalisation de l'export depuis 20261122T081323Z_export (déjà déplacé vers provenance/) — pas de nouveau travail par langue
2026-11-22T09:08:19 INFO aa          0 exportés (déjà calculés),  0 écarté(s), drapeaux {} 
2026-11-22T09:08:20 INFO af     131091 exportés (déjà calculés),  1 écarté(s), drapeaux {'language_mismatch': 17240, 'template_generated': 8930, 'near_duplicate': 8, 'boilerplate': 75} 
2026-11-22T09:08:20 INFO ak          1 exportés (déjà calculés),  0 écarté(s), drapeaux {'language_mismatch': 1} 
2026-11-22T09:08:20 INFO am      16286 exportés (déjà calculés),  0 écarté(s), drapeaux {'language_mismatch': 2037, 'template_generated': 2353, 'boilerplate': 13, 'near_duplicate': 14} 
2026-11-22T09:08:20 INFO ann       494 exportés (déjà calculés),  0 écarté(s), drapeaux {'language_mismatch': 25} 
2026-11-22T09:08:38 INFO ar    1332977 exportés (déjà calculés),  1 écarté(s), drapeaux {'language_mismatch': 346707, 'template_generated': 96339,

## Vérifications rapides

In [4]:
# Fichiers produits et tailles
for p in ["corpus", "provenance", "provenance.parquet", "exclusions.jsonl", "export_exemples.jsonl"]:
    chemin = ROOT / "data" / "processed" / p
    if chemin.is_dir():
        n = sum(1 for _ in chemin.glob("*.jsonl"))
        print(f"{p}/ : {n} fichiers")
    else:
        print(f"{p} : {chemin.stat().st_size / 1e6:.1f} Mo" if chemin.exists() else f"{p} : absent")

corpus/ : 69 fichiers
provenance/ : 69 fichiers
provenance.parquet : 332.8 Mo
exclusions.jsonl : 0.0 Mo
export_exemples.jsonl : 0.1 Mo


In [5]:
# Les textes écartés à l'export (erreurs de collecte)
excl_path = ROOT / "data" / "processed" / "exclusions.jsonl"
pd.read_json(excl_path, lines=True) if excl_path.exists() and excl_path.stat().st_size else "aucune exclusion"

,langue,text_sha256_collecte,raison
0,af,756f6b4e49037945604c58f58db5959caadbae88247dc7...,erreur de collecte
1,ar,e4cf3e8d047cccdb09e407ba560d7479c78834710f43d0...,erreur de collecte
2,ha,0fdcd4f2c341fd362aa7b4b1331cbe0cb8c34790625cb7...,erreur de collecte
3,ig,0fdcd4f2c341fd362aa7b4b1331cbe0cb8c34790625cb7...,erreur de collecte


In [6]:
# Quelques exemples (texte + provenance) pour relecture
pd.read_json(ROOT / "data" / "processed" / "export_exemples.jsonl", lines=True)

,text_sha256,language_code,source_url,source_dump,revision_date,collected_at,license,quality_flags,text_extrait
0,989ca9299cf255fc3313c33707998aeb11d5874cbbbcb8...,af,https://af.wikipedia.org/w/index.php?oldid=287...,afwiki-20261001-pages-articles,2026-02-06T20:50:36Z,2026-11-05 20:53:52+00:00,CC-BY-SA-4.0,[],Afrika / / Oppervlakte / 30 221 532 km² (2de)...
1,ebaf67c4bf34ad818017186d89d800cec8965d5736877a...,af,https://af.wikipedia.org/w/index.php?oldid=296...,afwiki-20261001-pages-articles,2026-09-22T10:50:27Z,2026-11-05 20:53:52+00:00,CC-BY-SA-4.0,[],Hier volg 'n lys van Afrikaanse skrywers: / /...
2,8d887112eff1a31670339b4c642fddec9ebe4b7ed9e9f9...,ak,https://ak.wikipedia.org/w/index.php?oldid=29906,akwiki-20261001-pages-articles,2023-05-17T06:21:17Z,2026-11-05 06:58:20+00:00,CC-BY-SA-4.0,[language_mismatch],This wiki has been locked (see discussion). To...
3,a7500e3b866678d2e083551e97407bc5bf731f302c8ddc...,am,https://am.wikipedia.org/w/index.php?oldid=386337,amwiki-20261001-pages-articles,2025-07-12T11:37:31Z,2026-11-05 08:46:26+00:00,CC-BY-SA-4.0,[],ለዕለቱ የተመረጠ ጽሑፍ ኒንቴንዶ ኔንቲዶ ኮ ሁለቱንም የቪዲዮ ጨዋታዎችን ...
4,bd3f75a0f2828b1b6fdb88f9a363d8dfc592e944db9dd3...,am,https://am.wikipedia.org/w/index.php?oldid=340201,amwiki-20261001-pages-articles,2017-07-16T17:03:34Z,2026-11-05 08:46:26+00:00,CC-BY-SA-4.0,[],የሕክምና ታሪክ በኢትዮጵያ / / ወቅታዊ ሕዝባዊ ጤና ጉዳዮች በኢትዮጵያ...
...,...,...,...,...,...,...,...,...,...
126,0289114de4d8c0658d042738ffc73c4d6696f664877eac...,yo,https://yo.wikipedia.org/w/index.php?oldid=473574,yowiki-20261001-pages-articles,2013-03-11T10:32:50Z,2026-11-05 20:41:11+00:00,CC-BY-SA-4.0,[language_mismatch],B / / el: βήτα / / ba bb bc bd be bf bg bh b...
127,c19fee9c7193b12b60b789a3ac9876725ed21cf339cdb4...,zgh,https://zgh.wikipedia.org/w/index.php?oldid=87980,api-20261105,2025-03-28T09:47:36Z,2026-11-05 08:40:51+00:00,CC-BY-SA-4.0,[],"ⴰⵎⴰⴳⵔⴰⴷ ⴰⴷ ⵉⴳⴰ ⵙⵓⵍ ⴰⴳⵓⵊⵉⵍ. ⵎⴽ ⵜⵓⴼⵉⴷ, ⵔⵏⵓ ⴰⵙⵖⵓⵏ..."
128,d5913ec6b3affe03e88ed99bbb2ed8ac98e1b261a76717...,zgh,https://zgh.wikipedia.org/w/index.php?oldid=17...,zghwiki-20261001-pages-articles,2026-05-04T13:52:07Z,2026-11-05 08:40:51+00:00,CC-BY-SA-4.0,[boilerplate],ⵜⴰⵔⴰⴽⴰⵍⵜ / ⵉⵙⴰⵖⵓⵍⵏ / ⴰⵙⴳⵣⴰⵡⴰⵍ ⵏ ⴰⵙⵉⵏⴰⴳ ⴰⴳⵍⴷⴰⵏ ...
129,1b67399e66fe40aa08bf46203981fdb144b57f1249bb6c...,zu,https://zu.wikipedia.org/w/index.php?oldid=126035,zuwiki-20261001-pages-articles,2026-07-13T11:06:42Z,2026-11-05 08:43:16+00:00,CC-BY-SA-4.0,[],"Siyakwamukela kuWikiphidiya yesiZulu, Umthombo..."


In [7]:
# Aperçu du registre de provenance final (Parquet)
pd.read_parquet(ROOT / "data" / "processed" / "provenance.parquet").sample(10, random_state=42)

,text_sha256,language_code,source_url,source_dump,revision_date,collected_at,license,quality_flags
3462519,0a6b9a616c6b168e99a61f1b520a38d97cd4b7089d4b1e...,ig,https://ig.wikipedia.org/w/index.php?oldid=177945,igwiki-20261001-pages-articles,2024-07-07T07:33:57Z,2026-11-05T20:52:06Z,CC-BY-SA-4.0,[]
3530918,2902924c1f4828b4771a2a28729e20673131064d186497...,mg,https://mg.wikipedia.org/w/index.php?oldid=921426,mgwiki-20261001-pages-articles,2018-10-29T11:25:00Z,2026-11-05T20:43:35Z,CC-BY-SA-4.0,[template_generated]
2183652,1b4465665f356af4b3e47a0a689f615f5e027d3fc4e843...,arz,https://arz.wikipedia.org/w/index.php?oldid=12...,arzwiki-20261001-pages-articles,2025-05-31T11:32:59Z,2026-11-05T21:09:26Z,CC-BY-SA-4.0,"[boilerplate, template_generated]"
3269536,4bea6406372ae5829cfcc5b10fb54c03d4371966d5430c...,fr,https://fr.wikipedia.org/w/index.php?oldid=227...,api-20261105,2025-08-08T12:13:31Z,2026-11-05T23:11:58Z,CC-BY-SA-4.0,[]
1322328,3583def481cfcb276b4bde1a3becc5ebcecb9313d07695...,ar,https://ar.wikipedia.org/w/index.php?oldid=711...,arwiki-20261001-pages-articles,2025-06-16T07:45:30Z,2026-11-05T22:12:31Z,CC-BY-SA-4.0,[language_mismatch]
2203942,76b00a41de43a9a10e8ae5485e9405519a50229d5bc129...,arz,https://arz.wikipedia.org/w/index.php?oldid=11...,arzwiki-20261001-pages-articles,2025-04-15T18:18:52Z,2026-11-05T21:09:39Z,CC-BY-SA-4.0,[]
2825693,4fc821410335e8faa314a5198a41dea58b1f67f82733f4...,arz,https://arz.wikipedia.org/w/index.php?oldid=11...,arzwiki-20261001-pages-articles,2025-04-30T11:00:37Z,2026-11-05T21:15:02Z,CC-BY-SA-4.0,[]
2889570,259578ea322226866be381e24f7ce47dacee856d8fbf45...,arz,https://arz.wikipedia.org/w/index.php?oldid=11...,arzwiki-20261001-pages-articles,2025-04-14T18:09:22Z,2026-11-05T21:15:45Z,CC-BY-SA-4.0,[]
625160,3619fe84f05c2947ee1903a4e69264357bef565297c9c6...,ar,https://ar.wikipedia.org/w/index.php?oldid=740...,arwiki-20261001-pages-articles,2026-03-19T01:44:38Z,2026-11-05T21:51:31Z,CC-BY-SA-4.0,[language_mismatch]
3060140,6af32cac22279adf5f26272980b215fad200d09ea1075e...,arz,https://arz.wikipedia.org/w/index.php?oldid=97...,arzwiki-20261001-pages-articles,2024-03-30T10:50:42Z,2026-11-05T21:16:33Z,CC-BY-SA-4.0,[]
